# Agentic RAG Pipeline
This notebook upgrades the static pipeline to an Agentic RAG pipeline with **Routing** and **Self-Correction**.

In [16]:
import os
from dotenv import load_dotenv
from groq import Groq
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Connect to the existing ChromaDB collection we built in the static pipeline!
chroma_client = chromadb.PersistentClient(path="./chroma_db")
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_collection(name="my_rag_collection", embedding_function=emb_fn)
print("Connected to ChromaDB!")


Connected to ChromaDB!


### Step 1: The Router
Decides whether to search our documents or just answer directly.

In [17]:
def route_query(question):
    print(f"--- [ROUTER] Analyzing query: '{question}' ---")
    prompt = f"""You are a query router. Decide if the following question requires retrieving external context from a knowledge base (like a resume, notes, or project README), or if it can be answered directly from general knowledge (like basic math, greetings, or common facts).
    
    Question: {question}
    
    Reply exactly with ONLY either 'needs_retrieval' or 'answer_directly'."""
    
    response = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b", 
        temperature=0.0
    )
    decision = response.choices[0].message.content.strip().lower()
    
    if "needs_retrieval" in decision:
        return "needs_retrieval"
    return "answer_directly"


### Step 2: Self-Check & Rewrite Tools
Checks if retrieved chunks are relevant, and rewrites the query if they aren't.

In [18]:
def check_relevance(question, context):
    prompt = f"""Do the following excerpts provide enough information to fully answer the question? 
    Reply exactly with 'YES' or 'NO'.
    
    Context:
    {context}
    
    Question:
    {question}"""
    
    response = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b", 
        temperature=0.0
    )
    decision = response.choices[0].message.content.strip().upper()
    return "YES" in decision

def rewrite_query(question):
    prompt = f"""The following question couldn't be answered by our initial search. Please rewrite the question to pull out different keywords or make it more searchable for a vector database.
    Original Question: {question}
    Reply with ONLY the rewritten question."""
    
    response = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b", 
        temperature=0.7
    )
    return response.choices[0].message.content.strip()


### Step 3: Agentic Pipeline
Putting it all together with a max of 2 retries.

In [19]:
def generate_answer(question, context=None):
    if context:
        prompt = f"Use the following context to answer the question.\n\nContext:\n{context}\n\nQuestion:\n{question}"
    else:
        prompt = f"Please answer the following question:\n\nQuestion:\n{question}"
        
    response = groq_client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ],
        model="openai/gpt-oss-120b", 
        temperature=0.2
    )
    return response.choices[0].message.content

def agentic_rag(original_question):
    print(f"\n{'='*50}")
    print(f"NEW TASK: {original_question}")
    print(f"{'='*50}")
    
    # 1. Routing
    route_decision = route_query(original_question)
    print(f"[LOG] Router Decision: {route_decision}")
    
    if route_decision == "answer_directly":
        print("[LOG] Skipping retrieval. Answering directly.")
        answer = generate_answer(original_question)
        print(f"\n--- FINAL ANSWER ---\n{answer}")
        return answer
        
    # 2. Retrieval with self-correction loop
    current_query = original_question
    max_retries = 2
    
    for attempt in range(max_retries + 1):
        print(f"\n[LOG] Retrieval Attempt {attempt + 1}/{max_retries + 1}")
        print(f"[LOG] Searching for: '{current_query}'")
        
        results = collection.query(
            query_texts=[current_query],
            n_results=4
        )
        retrieved_chunks = results['documents'][0]
        context = "\n\n".join(retrieved_chunks)
        
        print("[LOG] Retrieved context chunks.")
        
        # Self-Check
        is_relevant = check_relevance(original_question, context)
        print(f"[LOG] Self-Check Passed? {'YES' if is_relevant else 'NO'}")
        
        if is_relevant:
            print("[LOG] Context is good. Generating final answer.")
            answer = generate_answer(original_question, context)
            print(f"\n--- FINAL ANSWER ---\n{answer}")
            return answer
        else:
            if attempt < max_retries:
                print("[LOG] Context is not sufficient. Rewriting query...")
                current_query = rewrite_query(original_question)
            else:
                print("[LOG] Max retries reached. Generating best-effort answer with current context.")
                answer = generate_answer(original_question, context)
                print(f"\n--- FINAL ANSWER (Best Effort) ---\n{answer}")
                return answer


### Step 4: Testing the Agent
Let's see if it correctly routes direct questions vs. RAG questions, and if it retries bad queries!

In [43]:

agentic_rag("What is 2 + 2?")





NEW TASK: What is 2 + 2?
--- [ROUTER] Analyzing query: 'What is 2 + 2?' ---
[LOG] Router Decision: answer_directly
[LOG] Skipping retrieval. Answering directly.

--- FINAL ANSWER ---
2 + 2 = 4.


'2\u202f+\u202f2\u202f=\u202f4.'

In [44]:

agentic_rag("Experience of bikrant ?")


NEW TASK: Experience of bikrant ?
--- [ROUTER] Analyzing query: 'Experience of bikrant ?' ---
[LOG] Router Decision: needs_retrieval

[LOG] Retrieval Attempt 1/3
[LOG] Searching for: 'Experience of bikrant ?'
[LOG] Retrieved context chunks.
[LOG] Self-Check Passed? YES
[LOG] Context is good. Generating final answer.

--- FINAL ANSWER ---
Bikrant has **2 years of professional experience** as a software engineer, specializing in **Python, AI, and distributed systems**. He also enjoys building agentic workflows.


'Bikrant has **2\u202fyears of professional experience** as a software engineer, specializing in **Python, AI, and distributed systems**. He also enjoys building agentic workflows.'

In [45]:
# This will force a retry because resume doesn't mention a dog!
agentic_rag("What is the name of my dog?")



NEW TASK: What is the name of my dog?
--- [ROUTER] Analyzing query: 'What is the name of my dog?' ---
[LOG] Router Decision: needs_retrieval

[LOG] Retrieval Attempt 1/3
[LOG] Searching for: 'What is the name of my dog?'
[LOG] Retrieved context chunks.
[LOG] Self-Check Passed? NO
[LOG] Context is not sufficient. Rewriting query...

[LOG] Retrieval Attempt 2/3
[LOG] Searching for: 'What is the name of my pet dog?'
[LOG] Retrieved context chunks.
[LOG] Self-Check Passed? NO
[LOG] Context is not sufficient. Rewriting query...

[LOG] Retrieval Attempt 3/3
[LOG] Searching for: 'What is the name I gave to my pet dog?'
[LOG] Retrieved context chunks.
[LOG] Self-Check Passed? NO
[LOG] Max retries reached. Generating best-effort answer with current context.

--- FINAL ANSWER (Best Effort) ---
I’m sorry, but the information you provided doesn’t include any mention of a dog or its name. If you share that detail, I’d be happy to help you with any related questions!


'I’m sorry, but the information you provided doesn’t include any mention of a dog or its name. If you share that detail, I’d be happy to help you with any related questions!'